<a href="https://colab.research.google.com/github/M7office/Stroke/blob/main/AHA_stress_repair_profiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Slide 1 / Figure 1 — Stress- and repair-related protein profiles
# Single merged journal-style figure, Colab-ready script
#
# Input files expected in the current Colab folder (/content):
#   C_patient_data*.csv
#   C_NPX_data*.csv
#   strokecog_literature_aligned_pathway_framework*.csv
#   strokecog_literature_aligned_protein_pathway_assignment_summary*.csv
#
# Output folder:
#   AHA_slide01_outputs/
# ============================================================

from pathlib import Path
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -----------------------------
# User settings
# -----------------------------
BASE = Path.cwd()  # Colab default: /content
OUT = BASE / "AHA_slide01_outputs_v8"
OUT.mkdir(parents=True, exist_ok=True)

SIS3_CUTOFF = 63
RANDOM_SEED = 42
N_BOOT = 5000

PATHWAY_ASSIGNMENT_MODE = "all"  # "all" or "primary"
FILTER_LOW_DETECTION_PROTEINS = True
LOW_DETECTION_FILTER_MODE = "complete_case"

FIGURE_TITLE = "Stress- and Repair-Related Protein Profiles Show Potential to Differentiate Lower- and Higher-SIS3 Patients"

COLORS = {
    "lower": "#D55E00",
    "higher": "#0072B2",
    "effect": "#2E5C68",
    "black": "#303030",
    "gray": "#6E6E6E",
    "lightgray": "#D9D9D9",
    "verylight": "#EAEAEA",
}

MODULES = {
    "01": "Serotonin / tryptophan-kynurenine / monoamine metabolism",
    "02": "Vesicle secretion / extracellular vesicle / membrane trafficking",
    "03": "Cell death / cellular stress / proteostasis",
    "04": "Metabolic / lipid / atherosclerosis / mitochondrial-energy biology",
    "05": "mTOR / MAPK / NF-kB / growth-survival signaling",
    "06": "Hormone / neuroendocrine / HPA-like systemic signaling",
    "07": "Systemic organ injury / leakage / comorbidity markers",
    "08": "Peripheral immune / inflammatory activation",
    "09": "Complement / coagulation / platelet axis",
    "10": "Endothelial / BBB / neurovascular unit",
    "11": "Synaptic / neuronal plasticity / neurotrophic signaling",
    "12": "Integrin / ECM / cell adhesion / vascular remodeling",
}

PROFILES = {
    "Stress-response activation": [MODULES["01"], MODULES["02"], MODULES["03"], MODULES["05"]],
    "Repair / neurovascular remodeling": [MODULES["09"], MODULES["10"], MODULES["11"], MODULES["12"]],
}

PROFILE_ORDER = [
    "Stress-response activation",
    "Repair / neurovascular remodeling",
    "Stress–repair imbalance",
]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.8,
    "ytick.labelsize": 9.0,
    "legend.fontsize": 8.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 450,
})


def read_csv_safely(path: Path) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            df.columns = df.columns.astype(str).str.strip()
            return df
        except UnicodeDecodeError:
            pass
    df = pd.read_csv(path)
    df.columns = df.columns.astype(str).str.strip()
    return df


def csv_files():
    return sorted(BASE.glob("*.csv"))


def find_csv(contains_all=None, contains_any=None, required=True, label="file", exclude_any=None):
    contains_all = [x.lower() for x in (contains_all or [])]
    contains_any = [x.lower() for x in (contains_any or [])]
    exclude_any = [x.lower() for x in (exclude_any or [])]
    matches = []
    for f in csv_files():
        name = f.name.lower()
        if contains_all and not all(x in name for x in contains_all):
            continue
        if contains_any and not any(x in name for x in contains_any):
            continue
        if exclude_any and any(x in name for x in exclude_any):
            continue
        matches.append(f)
    if not matches:
        if required:
            raise FileNotFoundError(f"Could not find {label}. Tried all={contains_all}, any={contains_any}")
        return None
    return sorted(matches, key=lambda p: (len(p.name), p.name))[0]


def find_col(df, candidates, required=True, label="column", avoid=None):
    avoid = [a.lower() for a in (avoid or [])]
    lower_to_col = {c.lower(): c for c in df.columns}
    for cand in candidates:
        c = lower_to_col.get(cand.lower())
        if c is not None and not any(a in c.lower() for a in avoid):
            return c
    for c in df.columns:
        c_l = c.lower()
        if any(cand.lower() in c_l for cand in candidates):
            if not any(a in c_l for a in avoid):
                return c
    if required:
        raise ValueError(f"Could not find {label}. Tried {candidates}. Available columns: {df.columns.tolist()}")
    return None


def normalize_id_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.0$", "", s)
    return s


def normalize_oid_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    m = re.search(r"(OID\d+)", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    return s


def normalize_feature_column_name(c):
    s = str(c).strip()
    m = re.search(r"(OID\d+)", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    return s


def unique_preserve(seq):
    out, seen = [], set()
    for x in seq:
        if pd.isna(x):
            continue
        x = str(x).strip()
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out


def detect_time_years(series, colname=""):
    x = pd.to_numeric(series, errors="coerce")
    lname = str(colname).lower()
    max_val = np.nanmax(x.values) if np.isfinite(x).any() else np.nan
    if "year" in lname or "yrs" in lname or "yr" in lname:
        return x
    if "day" in lname or (np.isfinite(max_val) and max_val > 40):
        return x / 365.25
    if "month" in lname or (np.isfinite(max_val) and max_val > 6):
        return x / 12.0
    return x


def zscore_df(df):
    numeric = df.apply(pd.to_numeric, errors="coerce")
    return (numeric - numeric.mean(axis=0)) / numeric.std(axis=0, ddof=0).replace(0, np.nan)


def bootstrap_mean_diff_stats(a, b, n_boot=N_BOOT, seed=RANDOM_SEED):
    a = pd.Series(a).dropna().astype(float).values
    b = pd.Series(b).dropna().astype(float).values
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan, np.nan

    diff = float(np.mean(a) - np.mean(b))
    rng = np.random.default_rng(seed)

    boot = np.empty(n_boot)
    for i in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boot[i] = np.mean(aa) - np.mean(bb)
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    pooled_mean = (np.sum(a) + np.sum(b)) / (len(a) + len(b))
    a_null = a - np.mean(a) + pooled_mean
    b_null = b - np.mean(b) + pooled_mean
    boot_null = np.empty(n_boot)
    for i in range(n_boot):
        aa = rng.choice(a_null, size=len(a_null), replace=True)
        bb = rng.choice(b_null, size=len(b_null), replace=True)
        boot_null[i] = np.mean(aa) - np.mean(bb)
    p_boot = float(np.mean(np.abs(boot_null) >= abs(diff)))
    return diff, float(ci_low), float(ci_high), p_boot


def fmt_p3(p):
    if pd.isna(p):
        return "NA"
    return f"{float(p):.3f}"


def fmt_ci(diff, lo, hi):
    if pd.isna(diff) or pd.isna(lo) or pd.isna(hi):
        return "NA"
    return f"{diff:.2f} ({lo:.2f} to {hi:.2f})"


def wrap_caption(text, width=180):
    return "\n".join(textwrap.wrap(text, width=width, break_long_words=False))


print("Current folder:", BASE)
print("CSV files found:")
for f in csv_files():
    print(" -", f.name)

patient_file = find_csv(contains_any=["patient"], required=True, label="patient file")
npx_file = find_csv(contains_any=["npx"], required=True, label="NPX file")
framework_file = find_csv(contains_all=["pathway_framework"], required=True, label="pathway framework file")
assignment_file = find_csv(contains_all=["protein_pathway_assignment"], required=True, label="protein pathway assignment file")

patient = read_csv_safely(patient_file)
npx = read_csv_safely(npx_file)
framework = read_csv_safely(framework_file)
assign = read_csv_safely(assignment_file)

pathway_col_fw = find_col(framework, ["literature_aligned_pathway", "pathway"], label="framework pathway column")
framework_order = unique_preserve(framework[pathway_col_fw])
if not framework_order:
    raise ValueError("No pathway names found in pathway framework file.")

patient_id_col = find_col(
    patient,
    ["patient_id", "patientid", "participant_id", "subject_id", "sample_id", "record_id", "pid", "id"],
    required=False,
    label="patient ID column",
    avoid=["sis", "score", "time", "age"],
)
if patient_id_col is None:
    patient_id_col = patient.columns[0]

sis3_col = find_col(patient, ["sis3", "sis_3", "sis 3"], label="SIS3 column")
time_col = find_col(patient, ["time_year", "time_years", "years", "month", "day", "time_since", "timesince", "time"], label="time column")

patient = patient.copy()
patient["_patient_id_norm"] = patient[patient_id_col].map(normalize_id_value)
patient["sis3"] = pd.to_numeric(patient[sis3_col], errors="coerce")
patient["time_years"] = detect_time_years(patient[time_col], time_col)

ref_ids = set(patient["_patient_id_norm"].dropna().astype(str))
id_col_npx = None
best_overlap = -1
for c in npx.columns:
    vals = set(npx[c].map(normalize_id_value).dropna().astype(str))
    overlap = len(vals.intersection(ref_ids))
    if overlap > best_overlap:
        best_overlap = overlap
        id_col_npx = c

npx_value_col = find_col(npx, ["npx", "value"], required=False, label="NPX value column")
oid_col_npx = find_col(npx, ["oid", "olinkid", "assay", "protein_id"], required=False, label="OID column in NPX file")

if best_overlap > 0 and npx_value_col is not None and oid_col_npx is not None and id_col_npx != oid_col_npx:
    npx["_patient_id_norm"] = npx[id_col_npx].map(normalize_id_value)
    npx["_OID_norm"] = npx[oid_col_npx].map(normalize_oid_value)
    npx_wide = npx.pivot_table(index="_patient_id_norm", columns="_OID_norm", values=npx_value_col, aggfunc="mean").reset_index()
    merged = patient.merge(npx_wide, on="_patient_id_norm", how="inner")
elif best_overlap > 0:
    npx_wide = npx.rename(columns={c: normalize_feature_column_name(c) for c in npx.columns})
    id_col_norm = normalize_feature_column_name(id_col_npx)
    npx_wide["_patient_id_norm"] = npx_wide[id_col_norm].map(normalize_id_value)
    merged = patient.merge(npx_wide, on="_patient_id_norm", how="inner")
else:
    if len(npx) != len(patient):
        raise ValueError(
            "Could not match patient IDs, and NPX row count does not match patient row count. Please add a patient ID column to the NPX file."
        )
    npx_wide = npx.rename(columns={c: normalize_feature_column_name(c) for c in npx.columns})
    npx_wide["_row_order"] = np.arange(len(npx_wide))
    patient["_row_order"] = np.arange(len(patient))
    merged = patient.merge(npx_wide, on="_row_order", how="inner")
    merged["_patient_id_norm"] = merged["_patient_id_norm"].fillna(merged["_row_order"].astype(str))
    print("\nNote: NPX file has no matching patient ID column. Using row-order alignment because row counts match.")

if merged.empty:
    raise ValueError("Patient and NPX files did not merge.")

oid_col_assign = find_col(assign, ["OID", "oid", "olinkid", "protein_id"], label="OID column in assignment file")
if PATHWAY_ASSIGNMENT_MODE == "primary":
    pathway_col_assign = find_col(assign, ["primary_literature_aligned_pathway_draft", "primary_literature_aligned_pathway", "pathway"], label="primary pathway column")
    mapping = assign[[oid_col_assign, pathway_col_assign]].copy()
    mapping.columns = ["OID", "pathway"]
elif PATHWAY_ASSIGNMENT_MODE == "all":
    pathway_col_assign = find_col(assign, ["all_literature_aligned_pathways_draft", "all_literature_aligned_pathways", "all_pathways", "pathways"], label="all-pathways column")
    mapping = assign[[oid_col_assign, pathway_col_assign]].copy()
    mapping.columns = ["OID", "pathway"]
    mapping["pathway"] = mapping["pathway"].astype(str).str.split(r"\s*[;|,]\s*", regex=True)
    mapping = mapping.explode("pathway")
else:
    raise ValueError("PATHWAY_ASSIGNMENT_MODE must be 'primary' or 'all'.")

mapping["OID"] = mapping["OID"].map(normalize_oid_value)
mapping["pathway"] = mapping["pathway"].astype(str).str.strip()
mapping = mapping.dropna()
mapping = mapping[mapping["pathway"].isin(framework_order)]

protein_cols = [c for c in merged.columns if re.fullmatch(r"OID\d+", str(c), flags=re.IGNORECASE)]
protein_cols = [normalize_feature_column_name(c) for c in protein_cols]
protein_cols = [c for c in protein_cols if c in merged.columns]
protein_numeric = merged[protein_cols].apply(pd.to_numeric, errors="coerce")

if FILTER_LOW_DETECTION_PROTEINS and LOW_DETECTION_FILTER_MODE == "complete_case":
    detection_rate = protein_numeric.notna().sum(axis=0) / len(protein_numeric)
    retained_protein_cols = detection_rate[detection_rate == 1.0].index.tolist()
    print(f"\nProteins before complete-case filter: {len(protein_cols)}")
    print(f"Proteins retained after complete-case filter: {len(retained_protein_cols)}")
    protein_cols = retained_protein_cols
    protein_numeric = protein_numeric[protein_cols]
    mapping = mapping[mapping["OID"].isin(protein_cols)].copy()
else:
    mapping = mapping[mapping["OID"].isin(protein_cols)].copy()

if mapping.empty:
    raise ValueError("No pathway-assignment OIDs matched retained NPX protein columns.")

protein_z = zscore_df(protein_numeric)
score_df = merged[["_patient_id_norm", "sis3", "time_years"]].copy()
for pathway in framework_order:
    oids = sorted(set(mapping.loc[mapping["pathway"] == pathway, "OID"]).intersection(protein_z.columns))
    score_df[pathway] = protein_z[oids].mean(axis=1) if len(oids) else np.nan

score_df["sis3_level"] = np.where(score_df["sis3"] <= SIS3_CUTOFF, "Lower SIS3 (≤63)", "Higher SIS3 (>63)")
score_df.to_csv(OUT / "slide01_pathway_scores_used.csv", index=False)
mapping.to_csv(OUT / "slide01_protein_pathway_mapping_used.csv", index=False)

profile_membership = []
for profile_name, module_list in PROFILES.items():
    available = [m for m in module_list if m in score_df.columns]
    missing = [m for m in module_list if m not in score_df.columns]
    if len(available) == 0:
        raise ValueError(f"No modules available for profile: {profile_name}")
    score_df[profile_name] = score_df[available].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    for m in available:
        profile_membership.append({"profile": profile_name, "module": m, "module_available": True})
    for m in missing:
        profile_membership.append({"profile": profile_name, "module": m, "module_available": False})

score_df["Stress–repair imbalance"] = (
    pd.to_numeric(score_df["Stress-response activation"], errors="coerce")
    - pd.to_numeric(score_df["Repair / neurovascular remodeling"], errors="coerce")
)

pd.DataFrame(profile_membership).to_csv(OUT / "slide01_profile_membership.csv", index=False)
score_df.to_csv(OUT / "slide01_patient_level_profile_scores.csv", index=False)

rows = []
for i, profile in enumerate(PROFILE_ORDER):
    lower = pd.to_numeric(score_df.loc[score_df["sis3_level"] == "Lower SIS3 (≤63)", profile], errors="coerce").dropna()
    higher = pd.to_numeric(score_df.loc[score_df["sis3_level"] == "Higher SIS3 (>63)", profile], errors="coerce").dropna()
    diff, lo, hi, p = bootstrap_mean_diff_stats(lower, higher, seed=RANDOM_SEED + i)
    rows.append({
        "Profile": profile,
        "Lower SIS3, n": len(lower),
        "Higher SIS3, n": len(higher),
        "Lower SIS3, mean": float(lower.mean()),
        "Higher SIS3, mean": float(higher.mean()),
        "Mean difference, Lower - Higher": diff,
        "95% CI lower": lo,
        "95% CI upper": hi,
        "P value": p,
        "P value formatted": fmt_p3(p),
        "Mean difference (95% CI)": fmt_ci(diff, lo, hi),
    })

stats = pd.DataFrame(rows)
stats.to_csv(OUT / "slide01_profile_low_vs_high_statistics.csv", index=False)

# -----------------------------
# Make merged single figure
# -----------------------------
plot_labels = PROFILE_ORDER
n = len(plot_labels)
y = np.arange(n)
bar_height = 0.27

lower_means = [score_df.loc[score_df["sis3_level"] == "Lower SIS3 (≤63)", p].mean() for p in plot_labels]
higher_means = [score_df.loc[score_df["sis3_level"] == "Higher SIS3 (>63)", p].mean() for p in plot_labels]

effects = stats["Mean difference, Lower - Higher"].values
ci_lows = stats["95% CI lower"].values
ci_highs = stats["95% CI upper"].values
pvals = stats["P value formatted"].values

fig = plt.figure(figsize=(15.0, 6.9))
gs = GridSpec(3, 5, figure=fig, height_ratios=[0.46, 3.05, 1.62], width_ratios=[1.18, 0.10, 1.00, 0.56, 0.48], hspace=0.08, wspace=0.045)

title_ax = fig.add_subplot(gs[0, :])
title_ax.axis("off")
title_ax.text(0.00, 0.72, "Figure 1. " + FIGURE_TITLE, ha="left", va="center", fontsize=12.8, fontweight="bold", color=COLORS["black"], transform=title_ax.transAxes)

ax_bar = fig.add_subplot(gs[1, 0])
ax_gap = fig.add_subplot(gs[1, 1])
ax_forest = fig.add_subplot(gs[1, 2], sharey=ax_bar)
ax_est = fig.add_subplot(gs[1, 3], sharey=ax_bar)
ax_p = fig.add_subplot(gs[1, 4], sharey=ax_bar)
ax_gap.axis("off")
ax_est.axis("off")
ax_p.axis("off")



# Left: group means
ax_bar.barh(y - bar_height/2, lower_means, height=bar_height, color=COLORS["lower"], label="Lower SIS3 (≤63)")
ax_bar.barh(y + bar_height/2, higher_means, height=bar_height, color=COLORS["higher"], label="Higher SIS3 (>63)")
ax_bar.axvline(0, color=COLORS["black"], lw=0.8)
ax_bar.set_yticks(y)
ax_bar.set_yticklabels(plot_labels)
ax_bar.invert_yaxis()
ax_bar.set_xlabel("Mean standardized profile value")
ax_bar.set_title("Mean standardized profile values by SIS3 group", loc="left", pad=8, fontweight="bold")
ax_bar.tick_params(axis="both", length=3, color=COLORS["gray"])
ax_bar.set_axisbelow(True)
ax_bar.grid(axis="x", color=COLORS["verylight"], lw=0.6)
finite_bar = np.array([*lower_means, *higher_means], dtype=float)
finite_bar = finite_bar[np.isfinite(finite_bar)]
if len(finite_bar):
    bar_lim = max(0.35, np.max(np.abs(finite_bar)) + 0.08)
    bar_lim = np.ceil(bar_lim / 0.1) * 0.1
else:
    bar_lim = 0.4
ax_bar.set_xlim(-bar_lim, bar_lim)
ax_bar.legend(frameon=False, loc="lower right", handlelength=1.6, borderaxespad=0.3)

# Right: forest plot
ax_forest.hlines(y, ci_lows, ci_highs, color=COLORS["effect"], lw=1.4)
ax_forest.plot(effects, y, marker="s", linestyle="None", color=COLORS["effect"], markersize=5.0)
ax_forest.axvline(0, color=COLORS["black"], lw=0.8, ls=":")
ax_forest.set_xlabel("Mean difference in standardized profile value, lower − higher SIS3 (95% CI)")
ax_forest.set_title("Mean differences in standardized profile values", loc="left", pad=8, fontweight="bold")
ax_forest.tick_params(axis="x", length=3, color=COLORS["gray"])
ax_forest.tick_params(axis="y", left=False, labelleft=False)
ax_forest.set_axisbelow(True)
ax_forest.grid(axis="x", color=COLORS["verylight"], lw=0.6)

finite_x = np.array([*ci_lows, *ci_highs, *effects], dtype=float)
finite_x = finite_x[np.isfinite(finite_x)]
if len(finite_x):
    forest_lim = max(0.45, np.max(np.abs(finite_x)) + 0.08)
    forest_lim = np.ceil(forest_lim / 0.1) * 0.1
else:
    forest_lim = 0.5
ax_forest.set_xlim(-forest_lim, forest_lim)

# Estimate and P-value columns aligned to the shared y axis
est_labels = [fmt_ci(d, lo, hi) for d, lo, hi in zip(effects, ci_lows, ci_highs)]

ax_est.set_xlim(0, 1)
ax_est.text(0.18, 1.00, "Estimate (95% CI)", transform=ax_est.transAxes,
            ha="left", va="bottom", fontsize=8.8, fontweight="bold", color=COLORS["black"])
for yi, est in zip(y, est_labels):
    ax_est.text(0.18, yi, est, ha="left", va="center", fontsize=8.8, color=COLORS["black"])
ax_est.set_ylim(ax_bar.get_ylim())

ax_p.set_xlim(0, 1)
ax_p.text(0.00, 1.00, "P value", transform=ax_p.transAxes, ha="left", va="bottom", fontsize=8.8, fontweight="bold", color=COLORS["black"])
for yi, pv in zip(y, pvals):
    ax_p.text(0.00, yi, pv, ha="left", va="center", fontsize=8.8, color=COLORS["black"])
ax_p.set_ylim(ax_bar.get_ylim())

caption_ax = fig.add_subplot(gs[2, :])
caption_ax.axis("off")
caption = (
    "Lower SIS3 was defined as SIS3 ≤63 and higher SIS3 as SIS3 >63. Profile values were standardized summary values of "
    "hypothesis-selected protein modules. Stress-response activation included monoamine/kynurenine metabolism, vesicle trafficking, "
    "cellular stress/proteostasis, and growth-survival signaling. Repair/neurovascular remodeling included complement/coagulation, "
    "endothelial/BBB, synaptic/neuroplasticity, and integrin/ECM remodeling. Stress–repair imbalance was calculated as "
    "stress-response activation minus repair/neurovascular remodeling. Positive mean differences indicate higher values in lower-SIS3 "
    "patients. Error bars indicate bootstrap 95% CIs; P values were estimated by bootstrap resampling."
)
caption_ax.text(0.00, 0.68, wrap_caption(caption, width=150), ha="left", va="top", fontsize=8.7, color=COLORS["black"], linespacing=1.25, transform=caption_ax.transAxes)

fig.subplots_adjust(left=0.11, right=0.96, top=0.92, bottom=0.12)

for ext in ["png", "pdf", "svg"]:
    fig.savefig(OUT / f"figure1_stress_repair_profiles_merged_journal_style_v8.{ext}", bbox_inches="tight")
plt.close(fig)

print("\nCreated Slide 1 / Figure 1 outputs:")
for p in sorted(OUT.glob("figure1_stress_repair_profiles_merged_journal_style_v8.*")):
    print(" -", p)
print("\nStatistics table:")
print(stats[["Profile", "Lower SIS3, n", "Higher SIS3, n", "Mean difference (95% CI)", "P value formatted"]].to_string(index=False))


Current folder: /content
CSV files found:
 - C_NPX_data.csv
 - C_patient_data.csv
 - NAME_OID.csv
 - strokecog_literature_aligned_pathway_framework.csv
 - strokecog_literature_aligned_protein_pathway_assignment_summary.csv

Proteins before complete-case filter: 1196
Proteins retained after complete-case filter: 1011

Created Slide 1 / Figure 1 outputs:
 - /content/AHA_slide01_outputs_v3/figure1_stress_repair_profiles_merged_journal_style_v3.pdf
 - /content/AHA_slide01_outputs_v3/figure1_stress_repair_profiles_merged_journal_style_v3.png
 - /content/AHA_slide01_outputs_v3/figure1_stress_repair_profiles_merged_journal_style_v3.svg

Statistics table:
                          Profile  Lower SIS3, n  Higher SIS3, n Mean difference (95% CI) P value formatted
       Stress-response activation             17              68     0.17 (-0.05 to 0.39)             0.145
Repair / neurovascular remodeling             17              68    -0.16 (-0.37 to 0.06)             0.163
          Stress–rep